In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Constants
RANDOM_STATE = 1082
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [3]:
# Load the dataset
try:
    df = pd.read_csv("../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../Datasets/csv/spoofing-merged-gps-only.csv")
    exit()

In [4]:
# Data Preprocessing

df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train (benign only) and test (all)
X_train = X_scaled[y == 0]
X_test = X_scaled
y_test = y

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [5]:
# Define the Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=8):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

input_dim = X_train.shape[1]
encoding_dim = 8
autoencoder = Autoencoder(input_dim, encoding_dim)

In [6]:
# Training
num_epochs = 200
batch_size = 32
learning_rate = 1e-3

criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=learning_rate)

train_loader = torch.utils.data.DataLoader(X_train_tensor, batch_size=batch_size, shuffle=True)

autoencoder.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        output = autoencoder(batch)
        loss = criterion(output, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(X_train_tensor)
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}")

Epoch 1/200, Loss: 0.249635
Epoch 10/200, Loss: 0.039148
Epoch 20/200, Loss: 0.032146
Epoch 30/200, Loss: 0.029576
Epoch 40/200, Loss: 0.028107
Epoch 50/200, Loss: 0.027429
Epoch 60/200, Loss: 0.026804
Epoch 70/200, Loss: 0.026453
Epoch 80/200, Loss: 0.026231
Epoch 90/200, Loss: 0.026208
Epoch 100/200, Loss: 0.026109
Epoch 110/200, Loss: 0.026075
Epoch 120/200, Loss: 0.025999
Epoch 130/200, Loss: 0.025998
Epoch 140/200, Loss: 0.026018
Epoch 150/200, Loss: 0.025950
Epoch 160/200, Loss: 0.025949
Epoch 170/200, Loss: 0.025960
Epoch 180/200, Loss: 0.025889
Epoch 190/200, Loss: 0.025930
Epoch 200/200, Loss: 0.025893


In [7]:
# Evaluation
autoencoder.eval()
with torch.no_grad():
    reconstructed = autoencoder(X_test_tensor)
    mse = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1).cpu().numpy()

# Threshold: 95th percentile of benign errors
threshold = np.percentile(mse[y == 0], 95)
y_pred = (mse > threshold).astype(int)


In [8]:
# Results
print(f"Autoencoder Threshold: {threshold:.4f}")
print(f"Model Accuracy on Test Set: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Autoencoder Threshold: 0.0705
Model Accuracy on Test Set: 95.55%
Confusion Matrix:
[[2967  157]
 [   4  494]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      3124
           1       0.76      0.99      0.86       498

    accuracy                           0.96      3622
   macro avg       0.88      0.97      0.92      3622
weighted avg       0.97      0.96      0.96      3622

